# AffectLab — IEMOCAP speaker-independent text training

This notebook trains DeBERTa-v3-small on the five private, speaker-independent IEMOCAP folds stored in Cloud Storage. Run the four-class benchmark first. The six-class AffectLab experiment is separate and opt-in because IEMOCAP contains only 40 fear examples. Checkpoints and results remain in the private bucket.

In [ ]:
!nvidia-smi
import torch

assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > GPU'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
import subprocess

from google.colab import auth

PROJECT_ID = 'cat-behaviour-research'
BUCKET = 'affectlab-research-raluca-biras'
PROCESSED_GCS = f'gs://{BUCKET}/data/processed/iemocap-text-v1'
RUNS_GCS = f'gs://{BUCKET}/runs/iemocap-text'
auth.authenticate_user()
subprocess.run(['gcloud', 'config', 'set', 'project', PROJECT_ID], check=True)

In [ ]:
import base64
import os
import sys
from pathlib import Path

from google.colab import userdata

REPO_URL = 'https://github.com/ralucabiras/emotion-aware-role-play-model.git'
REPO_DIR = Path('/content/emotion-aware-role-play-model')
github_token = userdata.get('GITHUB_TOKEN')
if not github_token:
    raise RuntimeError('Add a read-only GITHUB_TOKEN in Colab Secrets and enable notebook access.')
basic_auth = base64.b64encode(f'x-access-token:{github_token}'.encode()).decode()
auth_option = f'http.extraHeader=Authorization: Basic {basic_auth}'
if not REPO_DIR.exists():
    subprocess.run(['git', '-c', auth_option, 'clone', REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), '-c', auth_option, 'pull', '--ff-only'], check=True)
del github_token, basic_auth, auth_option
os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO_DIR / 'requirements-ml.txt')], check=True)
from huggingface_hub import login

hf_token = userdata.get('HF_TOKEN')
if hf_token:
    login(token=hf_token, add_to_git_credential=False)

In [ ]:
DATA_ROOT = Path('/content/iemocap-text-v1')
OUTPUT_ROOT = Path('/content/iemocap-runs')
subprocess.run(['gcloud', 'storage', 'rsync', '--recursive', PROCESSED_GCS, str(DATA_ROOT)], check=True)
assert (DATA_ROOT / 'manifest.json').exists(), 'Processed IEMOCAP manifest was not downloaded'
print('Processed folds ready:', DATA_ROOT)

## Four-class benchmark

Start with fold 5 as a pipeline check. After it succeeds, set `RUN_ALL_FOLDS = True` and rerun this cell. Existing fold 5 output may be reproducibly overwritten. Results are uploaded after every fold so a disconnected runtime does not lose completed work.

In [ ]:
SEED = 42
RUN_ALL_FOLDS = False
folds = [1, 2, 3, 4, 5] if RUN_ALL_FOLDS else [5]
benchmark_config = REPO_DIR / 'configs' / 'iemocap_benchmark4_deberta_v3_small.json'
benchmark_experiment = 'iemocap_benchmark4_deberta_v3_small'
for fold in folds:
    print(f'\n=== Benchmark fold {fold} ===')
    subprocess.run(
        [sys.executable, '-m', 'ml.training.train_iemocap_text', '--config', str(benchmark_config),
         '--data-root', str(DATA_ROOT), '--output-root', str(OUTPUT_ROOT), '--fold', str(fold), '--seed', str(SEED)],
        check=True,
    )
    subprocess.run(
        ['gcloud', 'storage', 'rsync', '--recursive', str(OUTPUT_ROOT / benchmark_experiment / f'fold-{fold}'),
         f'{RUNS_GCS}/{benchmark_experiment}/fold-{fold}'],
        check=True,
    )

In [ ]:
if RUN_ALL_FOLDS:
    experiment_dir = OUTPUT_ROOT / benchmark_experiment
    subprocess.run([sys.executable, '-m', 'ml.evaluation.summarize_iemocap_folds', str(experiment_dir)], check=True)
    summary = __import__('json').loads((experiment_dir / 'summary.json').read_text())
    display(summary)
    subprocess.run(['gcloud', 'storage', 'cp', str(experiment_dir / 'summary.json'), f'{RUNS_GCS}/{benchmark_experiment}/summary.json'], check=True)
else:
    metrics = __import__('json').loads((OUTPUT_ROOT / benchmark_experiment / 'fold-5' / 'metrics.json').read_text())
    display(metrics['validation_metrics'])
    display(metrics['test_metrics'])
    display(metrics['test_per_class'])

## Optional AffectLab six-class experiment

Run only after the complete four-class benchmark is preserved. Balanced inverse-frequency weights are capped at 10 to prevent the tiny anxiety class from dominating optimization. Report anxiety support and confidence intervals; do not present this task as proof of a reliable anxiety detector.

In [ ]:
RUN_AFFECTLAB_6 = False
if RUN_AFFECTLAB_6:
    affect_config = REPO_DIR / 'configs' / 'iemocap_affectlab6_deberta_v3_small.json'
    affect_experiment = 'iemocap_affectlab6_deberta_v3_small_weighted'
    for fold in range(1, 6):
        print(f'\n=== AffectLab six-class fold {fold} ===')
        subprocess.run(
            [sys.executable, '-m', 'ml.training.train_iemocap_text', '--config', str(affect_config),
             '--data-root', str(DATA_ROOT), '--output-root', str(OUTPUT_ROOT), '--fold', str(fold), '--seed', str(SEED)],
            check=True,
        )
        subprocess.run(
            ['gcloud', 'storage', 'rsync', '--recursive', str(OUTPUT_ROOT / affect_experiment / f'fold-{fold}'),
             f'{RUNS_GCS}/{affect_experiment}/fold-{fold}'],
            check=True,
        )
    affect_dir = OUTPUT_ROOT / affect_experiment
    subprocess.run([sys.executable, '-m', 'ml.evaluation.summarize_iemocap_folds', str(affect_dir)], check=True)
    affect_summary = __import__('json').loads((affect_dir / 'summary.json').read_text())
    display(affect_summary)
    subprocess.run(['gcloud', 'storage', 'cp', str(affect_dir / 'summary.json'), f'{RUNS_GCS}/{affect_experiment}/summary.json'], check=True)

## Completion criteria

- Five fold directories exist in the private bucket.
- `summary.json` reports fold mean/std and pooled metrics.
- Pooled test support equals 5,531 for `benchmark_4`.
- Do not choose a fold as the final model because it scored highest.
- Keep checkpoints, predictions, and processed IEMOCAP rows private.